#What will cleaning produce? For each table, we want accepted records, records needing review with a reason, and a count of extra identical copies discarded.

## Customers Cleaning

Prepare customer records for use in Sales. Keep one copy of identical records and send incomplete or conflicting records for review.

### Read and Standardize Customer Data

Read the five business fields from Bronze. Trim outer spaces and convert empty text to null.

In [0]:
from pyspark.sql import functions as F
#show only business columns 
customer_columns =  ["customer_id", "first_name", "last_name", "gender", "city"]
#read raw data with only the five business columns 
customer_raw = (
    spark.table("sales_quality.bronze_layer.customers")
    .select(*customer_columns))

#Create new data frame for cleaned data 
customers_prepared = customer_raw

#count original data as we can check our resulsts at the end 
source_count = customer_raw.count()

#iterate through our dataset to remove spaces 
for column in customer_columns:
    #this is spark column expression will be used later and attached to column of dataframe 
    trimmed = F.trim(F.col(column))
    #now we need to apply this expression to our columns with withColumns
    #so we choose column and apply the exresssion on and then check the vlaue if it's empty return null other with return the trimeed value 
    customers_prepared = customers_prepared.withColumn(
        column,
        F.when(trimmed != "",trimmed)
    )

print(f"Customers read from Bronze: {source_count}")


Customers read from Bronze: 49


### Keep One Copy of Each Customer Record

Compare all five business fields after standardization. Keep one copy of each identical record and count the extra copies discarded.

In [0]:
#we drop duplicate record when the five fields are identical 
customers_distinct = customers_prepared.dropDuplicates(
    customer_columns
)

#count how many extra copies we have 
copies_discarded = source_count - customers_distinct.count()

print(f"Extra identical copies discarded: {copies_discarded}")

Extra identical copies discarded: 0


### Identify Records That Need Review

Check each remaining record for missing fields. Flag every version of a populated customer ID that still appears more than once.

In [0]:
from pyspark.sql.window import Window
by_customer_id = Window.partitionBy("customer_id")

customers_checked = customers_distinct.withColumn(
    "rows_with_this_id",
    F.count("*").over(by_customer_id)
)

#let's add another column that would check review reasons 
# Build a review reason for each missing field.
reasons = []
for column in customer_columns:
    reasons.append(
        F.when(F.col(column).isNull(), F.lit(f"Missing {column}"))
    )

# After identical copies are removed, a repeated ID means details differ.
conflicting_id = (
    F.col("customer_id").isNotNull()
    & (F.col("rows_with_this_id") > 1)
)

reasons.append(
    F.when(conflicting_id, F.lit("Same customer ID has different details"))
)

# Join applicable reasons into one message. No reasons gives an empty string.
customers_checked = customers_checked.withColumn(
    "review_reason",
    F.concat_ws("; ", *reasons)
)
#if you displayed customers_checked this column of review_reasons would be empty is there is no reason for the record 

### Split Customers and Check the Counts

Accept records with no review reason. Keep the remaining records with their reasons and confirm that all source rows are accounted for.

In [0]:
#accepted customers records
customers_clean = (
    customers_checked
    .filter(F.col("review_reason")== "")
    .select(*customer_columns)
)

#rejected customers records
customers_review  = (
    customers_checked
    .filter(F.col("review_reason") != "")
    .select(*customer_columns, "review_reason")
)

#let's check counts 
clean_count = customers_clean.count()
review_count = customers_review.count()

# Each input row should have exactly one outcome.
if clean_count + review_count + copies_discarded != source_count:
    raise ValueError("Customer counts do not reconcile.")

print(f"Source rows:      {source_count}")
print(f"Accepted:         {clean_count}")
print(f"Need review:      {review_count}")
print(f"Copies discarded: {copies_discarded}")

display(customers_clean.orderBy("customer_id"))
display(customers_review.orderBy("customer_id"))

Source rows:      49
Accepted:         49
Need review:      0
Copies discarded: 0


customer_id,first_name,last_name,gender,city
CS_All_50,Allison,Connie,F,Albuquerque
CS_Ash_36,Ashley,Emma,F,Houston
CS_Ate_41,Atencio,Morris,M,Detroit
CS_Bar_13,Barajas,John,M,Columbus
CS_Bea_11,Beard,Robert,M,Indianapolis
CS_Bel_16,Bell,Thomas,M,Charlotte
CS_Bel_43,Bell,Angelo,M,Oklahoma City
CS_Bry_48,Bryant,John,M,Oklahoma City
CS_Buf_8,Buford,Ralph,M,Oklahoma City
CS_Bur_28,Burton,Theodore,M,Miami


customer_id,first_name,last_name,gender,city,review_reason


## Products Cleaning

Prepare products for use in Sales. Keep one copy of identical records, apply known category corrections, and check product IDs, names, prices and categories.

### Read Products and Prepare the Business Fields

Read the four product fields from Bronze. Trim outer spaces, convert blank text to null and count extra identical copies.

In [0]:
# Keep only the four product business fields.
product_columns = [
    "product_id",
    "product_name",
    "product_price",
    "product_category",
]
#read raw data with only the four business columns 
products_raw = (
    spark.table("sales_quality.bronze_layer.products")
    .select(*product_columns)
)
#get the count of products 
product_source_count = products_raw.count()

# Standardize text before comparing records.
products_prepared = products_raw 

for column in product_columns:
    trimmed = F.trim(F.col(column))
    products_prepared = products_prepared.withColumn(
        column,
        F.when(trimmed != "",trimmed)
    )

#keep only one copy when all four business fields 
products_distinct = products_prepared.dropDuplicates(
    product_columns
)
product_copies_discarded = product_source_count - products_distinct.count()

print(f"Products read from Bronze: {product_source_count}")
print(f"Source products: {product_source_count}")
print(f"Extra copies discarded: {product_copies_discarded}")


Products read from Bronze: 100
Source products: 100
Extra copies discarded: 0


### Apply Known Category Corrections

Look up each stored category in the alias table. Keep the original and proposed category side by side so we can see exactly what changed.

In [0]:
#the alias table contains the two approved spelling correction
aliases = (
    spark.table("sales_quality.bronze_layer.category_aliases")

.select(
    F.col("raw_value").alias("product_category"),
    F.col("correct_value").alias("suggested_category"),
)
)

#let's do join to check wich records we need to correct 
products_corrected = products_distinct.join(
    aliases,
    on = "product_category",
    how = "left"
)

# Use the suggestion when one exists. Otherwise keep the stored category.
products_corrected = products_corrected.withColumn(
    "clean_category",
    F.coalesce(F.col("suggested_category"), F.col("product_category")),
)

# Show the products for which a correction was available.
display(
    products_corrected
    .filter(F.col("suggested_category").isNotNull())
    .select("product_id", "product_category", "clean_category")
    .orderBy("product_id")
)

product_id,product_category,clean_category
P10001,electronics,Electronics
P10043,Clothng,Clothing


### Check the Prepared Products

Check that each product has an ID, a name, a finite positive price and an approved category. Flag repeated product IDs for review.

In [0]:
### Check the Prepared Products
# A matching row in this lookup confirms the final category is approved.
approved_categories = (
    spark.table("sales_quality.bronze_layer.product_categories")
    .select(F.col("category_name").alias("clean_category"))
    .distinct()
    .withColumn("category_approved", F.lit(True))
)

products_checked = products_corrected.join(
    approved_categories,
    on="clean_category",
    how="left",
)

# Keep the original price for review and create a number for accepted products.
products_checked = products_checked.withColumn(
    "numeric_price",
    F.expr("try_cast(product_price AS DOUBLE)"),
)

# Apply the same price rule we used during validation.
price = F.col("numeric_price")
valid_price = (
    price.isNotNull()
    & ~F.isnan(price)
    & (price > 0)
    & (price < F.lit(float("inf")))
)

# Count the records for each ID after identical copies were removed.
products_checked = products_checked.withColumn(
    "rows_with_this_id",
    F.count("*").over(Window.partitionBy("product_id")),
)

# Record every applicable reason in one message.
reasons = [
    F.when(F.col("product_id").isNull(), "Missing product_id"),
    F.when(F.col("product_name").isNull(), "Missing product_name"),
    F.when(F.col("product_price").isNull(), "Missing product_price"),
    F.when(
        F.col("product_price").isNotNull() & ~valid_price,
        "Invalid product price",
    ),
    F.when(F.col("clean_category").isNull(), "Missing product_category"),
    F.when(
        F.col("clean_category").isNotNull()
        & F.col("category_approved").isNull(),
        "Category has no approved match",
    ),
    F.when(
        F.col("product_id").isNotNull()
        & (F.col("rows_with_this_id") > 1),
        "Same product ID has different records",
    ),
]

# concat_ws skips null reasons. An empty message means all checks passed.
products_checked = products_checked.withColumn(
    "review_reason",
    F.concat_ws("; ", *reasons),
)

display(products_checked)

clean_category,product_category,product_id,product_name,product_price,suggested_category,category_approved,numeric_price,rows_with_this_id,review_reason
Electronics,electronics,P10001,Air purifier,66,Electronics,true,66.0,1,
Electronics,Electronics,P10002,Air conditioner,74,null,true,74.0,1,
Electronics,Electronics,P10003,Alarm clock,71,null,true,71.0,1,
Electronics,Electronics,P10004,Backup charger,80,null,true,80.0,1,
Electronics,Electronics,P10005,Bread maker,69,null,true,69.0,1,
Electronics,Electronics,P10006,Banknote counter,74,null,true,74.0,1,
Electronics,Electronics,P10007,Blender,63,null,true,63.0,1,
Electronics,Electronics,P10008,Bluetooth speaker,80,null,true,80.0,1,
Electronics,Electronics,P10009,Clothes dryer,72,null,true,72.0,1,
Electronics,Electronics,P10010,Coffee maker,78,null,true,78.0,1,


### Split Products and Check the Counts

Keep accepted products with their numeric prices and approved categories. Keep unresolved products with their original values and review reasons. Reconcile the counts with Bronze.

In [0]:
# Accepted products use the final category and numeric price.
products_clean = (
    products_checked
    .filter(F.col("review_reason") == "")
    .select(
        "product_id",
        "product_name",
        F.col("numeric_price").alias("product_price"),
        F.col("clean_category").alias("product_category"),
    )
)

# Review records retain the source fields so we can inspect the problem.
products_review = (
    products_checked
    .filter(F.col("review_reason") != "")
    .select(*product_columns, "clean_category", "review_reason")
)

product_clean_count = products_clean.count()
product_review_count = products_review.count()

# Every source record must be accounted for.
if product_clean_count + product_review_count + product_copies_discarded != product_source_count:
    raise ValueError("Product counts do not reconcile.")

print(f"Source products:   {product_source_count}")
print(f"Accepted:          {product_clean_count}")
print(f"Need review:       {product_review_count}")
print(f"Copies discarded:  {product_copies_discarded}")

display(products_clean.orderBy("product_id").limit(10))
display(products_review.orderBy("product_id"))

Source products:   100
Accepted:          99
Need review:       1
Copies discarded:  0


product_id,product_name,product_price,product_category
P10001,Air purifier,66.0,Electronics
P10002,Air conditioner,74.0,Electronics
P10003,Alarm clock,71.0,Electronics
P10004,Backup charger,80.0,Electronics
P10005,Bread maker,69.0,Electronics
P10006,Banknote counter,74.0,Electronics
P10007,Blender,63.0,Electronics
P10008,Bluetooth speaker,80.0,Electronics
P10009,Clothes dryer,72.0,Electronics
P10010,Coffee maker,78.0,Electronics


product_id,product_name,product_price,product_category,clean_category,review_reason
P10014,oven gloves,35,Unknown Category,Unknown Category,Category has no approved match


## Stores Cleaning

Use the approved location reference to correct store details. Keep accepted stores in `stores_clean` and unresolved records in `stores_review`.

### Read Stores and Remove Identical Copies

Trim outer spaces, convert blank text to null and keep one copy of each identical store record.

In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 
store_columns = ['store_id',"city","state","location"]
#Read the four business fields from Bronze
# Read the four business fields from Bronze.
stores_raw = (
    spark.table("sales_quality.bronze_layer.stores")
    .select(*store_columns)
)

store_source_count = stores_raw.count()
# Trim outer spaces. Internal spaces in cities and store IDs are preserved.
stores_prepared = stores_raw

for column in store_columns:
    trimmed = F.trim(F.col(column))
    stores_prepared = stores_prepared.withColumn(
        column,
        F.when(trimmed != "", trimmed)
    )

# Keep one copy only when all four business fields match.
stores_distinct = stores_prepared.dropDuplicates(store_columns)
store_copies_discarded = store_source_count - stores_distinct.count()
store_source_count = stores_raw.count()

# A repeated ID after this step means the store records have different details.
stores_distinct = stores_distinct.withColumn(
    "rows_with_this_id",
    F.count("*").over(Window.partitionBy("store_id")),
)

print(f"Source stores: {store_source_count}")
print(f"Extra copies discarded: {store_copies_discarded}")

Source stores: 14
Extra copies discarded: 0


### Look Up the Approved Location

Match each store to the reference by city. Use the reference state and location only when the city has exactly one complete approved combination.

In [0]:
#reference table we will rely on when returning verified values 
reference_locations = (
    spark.table("sales_quality.bronze_layer.store_locations")
    .select("city", "state", "location")
    .distinct()
)

city_lookup = (
    reference_locations
    .groupBy("city")
    .agg(
        F.count("*").alias("reference_count"),
        F.first("state").alias("reference_state"),
        F.first("location").alias("reference_location"),
    )
)

#then we join this city_lookup with stores locations . we can join it directly with reference table this only if we are sure that reference table there is only one record for each city 
# that's why we did create this city_lookup table 

# Keep every store, including stores whose city has no reference match.
stores_checked = stores_distinct.join(city_lookup, on="city", how="left")

usable_reference = (
    (F.col("reference_count") == 1)
    & F.col("reference_state").isNotNull()
    & (F.trim(F.col("reference_state")) != "")
    & F.col("reference_location").isNotNull()
    & (F.trim(F.col("reference_location")) != "")
)

# Keep the original fields alongside the proposed values for inspection.
stores_checked = (
    stores_checked
    .withColumn("clean_state", F.when(usable_reference, F.col("reference_state")))
    .withColumn("clean_location", F.when(usable_reference, F.col("reference_location")))
)



### Split Accepted Stores and Review Records

Accept stores with a populated, unique ID and one usable city match. Keep unresolved stores with their original details and a review reason.

In [0]:
# Give each unresolved store a reason. A row can have more than one.
reasons = [
    F.when(F.col("store_id").isNull(), "Missing store_id"),
    F.when(
        F.col("store_id").isNotNull() & (F.col("rows_with_this_id") > 1),
        "Same store ID has different records",
    ),
    F.when(F.col("city").isNull(), "Missing city"),
    F.when(
        F.col("city").isNotNull() & F.col("reference_count").isNull(),
        "City has no approved match",
    ),
    F.when(F.col("reference_count") > 1, "City has multiple approved combinations"),
    F.when(
        (F.col("reference_count") == 1) & ~usable_reference,
        "Reference state or location is missing",
    ),
]

# An empty review reason means the store passed the checks.
stores_checked = stores_checked.withColumn(
    "review_reason",
    F.concat_ws("; ", *reasons),
)

accepted_stores = stores_checked.filter(F.col("review_reason") == "")

# Accepted records use the approved state and location.
stores_clean = accepted_stores.select(
    "store_id",
    "city",
    F.col("clean_state").alias("state"),
    F.col("clean_location").alias("location"),
)

# Review records retain the stored values for investigation.
stores_review = (
    stores_checked
    .filter(F.col("review_reason") != "")
    .select(*store_columns, "review_reason")
)

### Inspect Corrections and Reconcile Counts

Display the accepted stores whose state or location changed. Check that accepted rows, review rows and discarded copies account for every source row.

In [0]:

# Compare stored and cleaned values. eqNullSafe also handles missing values.
store_corrections = (
    accepted_stores
    .filter(
        ~F.col("state").eqNullSafe(F.col("clean_state"))
        | ~F.col("location").eqNullSafe(F.col("clean_location"))
    )
    .select("store_id", "city", "state", "clean_state", "location", "clean_location")
)

display(store_corrections.orderBy("city"))

store_clean_count = stores_clean.count()
store_review_count = stores_review.count()

# Corrected stores are already included in the accepted count.
accounted_for = store_clean_count + store_review_count + store_copies_discarded

if accounted_for != store_source_count:
    raise ValueError("Store counts do not reconcile.")

print(f"Source stores:    {store_source_count}")
print(f"Accepted:         {store_clean_count}")
print(f"Need review:      {store_review_count}")
print(f"Copies discarded: {store_copies_discarded}")

display(stores_review.orderBy("store_id"))

store_id,city,state,clean_state,location,clean_location
St_Atl_Ge,Atlanta,Florida,Georgia,East Coast,East Coast
St_Mia_Fl,Miami,Florida,Florida,Midwest,East Coast


Source stores:    14
Accepted:         14
Need review:      0
Copies discarded: 0


store_id,city,state,location,review_reason
